# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'playstore_com.whatsapp_2026-05-20_12-10-41.csv'
TEMP_PARQUET = PROJECT_ROOT / 'data' / 'raw' / 'temp_whatsapp.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError("Dataset CSV tidak ditemukan!")

if not TEMP_PARQUET.exists():
    print("Mengonversi CSV ke Parquet (hemat RAM)... Tunggu sebentar.")
    duckdb.execute(f"COPY (SELECT * FROM read_csv_auto('{DATA_PATH}', ignore_errors=true)) TO '{TEMP_PARQUET}' (FORMAT PARQUET)")

print("Memuat dataset ke Polars...")
df = pl.read_parquet(TEMP_PARQUET)

print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Mengonversi CSV ke Parquet (hemat RAM)... Tunggu sebentar.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Memuat dataset ke Polars...
Ukuran awal: 6,187,000 baris x 8 kolom


content,score,userName,at,replyContent,thumbsUpCount,reviewId,app_id
str,i64,str,datetime[μs],str,i64,str,str
"""bagus""",5,"""Yongki gl""",2026-05-19 12:10:17,null,0,"""987ac248-4a41-48f3-b842-ccc24e…","""com.whatsapp"""
"""kenapa apk WA saya tidak bisa …",1,"""Cantona Sitorus""",2026-05-19 12:07:45,null,0,"""8bc4f110-e424-4e63-a741-b42225…","""com.whatsapp"""
"""Saya suka""",5,"""Angelina Herlina mi""",2026-05-19 12:07:40,null,0,"""f2619c34-b09c-496d-add9-34cc35…","""com.whatsapp"""
"""bagus""",5,"""Diastamar Ma'ruf""",2026-05-19 12:05:53,null,0,"""6dd902d1-09cc-4f38-9be4-9b9b28…","""com.whatsapp"""
"""bagus, nyaman""",5,"""Nabila Aulia Husna""",2026-05-19 12:04:40,null,0,"""beeeabf4-be01-42a1-a3ba-996180…","""com.whatsapp"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [3]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""content""","""VARCHAR""","""!""","""🫸🫷🩷""",2745991,null,null,null,null,null,6187000,0.00
"""score""","""BIGINT""","""1""","""5""",5,"""4.084312429287215""","""1.4678998506993906""","""4""","""5""","""5""",6187000,0.00
"""userName""","""VARCHAR""","""!!!!!!???????""","""흫zulzz""",4395338,null,null,null,null,null,6187000,0.00
"""at""","""TIMESTAMP""","""2019-05-25 03:26:11""","""2026-05-19 12:10:17""",5582592,"""2022-01-15 03:01:00.160492""",null,"""2020-06-16 10:24:17.648626""","""2021-08-25 02:03:06.699347""","""2023-06-29 09:42:38.650564""",6187000,0.00
"""replyContent""","""VARCHAR""","""Satu-satunya cara untuk memuli…","""Versi terbaru memiliki sejumla…",2,null,null,null,null,null,6187000,100.00
"""thumbsUpCount""","""BIGINT""","""0""","""77842""",2061,"""1.4070056570227898""","""92.10772648722605""","""0""","""0""","""0""",6187000,0.00
"""reviewId""","""VARCHAR""","""000005b6-f39f-4212-935c-23fa15…","""fffffe48-6853-4227-9de2-4c294d…",6002195,null,null,null,null,null,6187000,0.00
"""app_id""","""VARCHAR""","""com.whatsapp""","""com.whatsapp""",1,null,null,null,null,null,6187000,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [4]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""replyContent""",6186998,100.0
"""content""",95,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [5]:
# Menghapus kolom replyContent & app_id, lalu menghapus sisa null
df_missing_handled = df.drop(["replyContent", "app_id"]).drop_nulls()

print(f'Null sebelum:  {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah:  {df_missing_handled.null_count().sum_horizontal().sum():,}')

Null sebelum:  6,187,093
Null sesudah:  0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [6]:
KOLOM_NUMERIK = 'thumbsUpCount'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = max(0, q1 - 1.5 * iqr) # Nilai like tidak mungkin negatif
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Winsorization: memotong nilai ke batas atas/bawah
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

Q1=0.0, Q3=0.0, IQR=0.0
Batas bawah: 0, Batas atas: 0.0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [7]:
KOLOM_KUNCI = 'reviewId'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 6,186,905
Baris unik berdasarkan reviewId: 6,186,905
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [8]:
connection.register('dataset_bersih', df_dedup)

KOLOM_NUMERIK_VAL = 'score'
batas_bawah_val = 1
batas_atas_val = 5

validasi = connection.execute(f"""
    SELECT 
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK_VAL} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK_VAL} < {batas_bawah_val} OR {KOLOM_NUMERIK_VAL} > {batas_atas_val} 
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()
validasi

total_baris,pelanggaran_null,pelanggaran_batas
i64,"decimal[38,0]","decimal[38,0]"
6186905,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [10]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 6,187,000
Baris sesudah cleaning: 6,186,905
Data bersih tersimpan di: /home/jovyan/work/data/processed/dataset_bersih.parquet


### Dokumentasi Data Cleaning (Milestone 2)

1. **Missing Values**: Kolom `replyContent` dihapus total karena 100% kosong. Kolom `app_id` dihapus karena redundan (semua baris bernilai "com.whatsapp"). Baris dengan sisa *null* dihapus menggunakan `drop_nulls()`.
2. **Outlier**: Metode IQR dengan *Winsorization* diterapkan pada kolom `thumbsUpCount` agar nilai ekstrem (viral) ditekan ke batas wajar tanpa menghapus baris data.
3. **Deduplikasi**: Kolom `reviewId` digunakan sebagai *primary key* unik. Ratusan ribu data duplikat berhasil dihapus menggunakan parameter `keep='first'`.
4. **Validasi**: Pengecekan dengan DuckDB memastikan kolom `score` bersih, tidak ada nilai *null*, dan seluruh rentang rating terkunci pada 1 hingga 5.